# Attaching a fragment made of many residues

The fragment is the 31-residue semaglutide peptide, taken as a `Chain`
whose children are bonded `Residue` objects. It is attached through the
alpha carbon of its last glycine to lysine 48 of ubiquitin. Every fragment
residue keeps its name, its atoms and its formal charge.

In [ ]:
import logging, warnings
from rdkit import RDLogger

warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)
RDLogger.DisableLog("rdApp.*")

In [ ]:
from mbuild.biopolymers import Protein

mbuild_fragment_chain = Protein("../semaglutide_apo.pdb").chains[0]
print(len(mbuild_fragment_chain.children), "residues:", [(r.name, r.resnum) for r in mbuild_fragment_chain.children][:5], "...")

mbuild_protein = Protein("../1ubq_protonated.pdb")
mbuild_protein.deprotonate(48, "NZ")
mbuild_bond = mbuild_protein.attach(mbuild_fragment_chain, fragment_atom_name="CA", fragment_resnum=31, resnum=48, atom_name="NZ", relax=False)

In [ ]:
residues = list(mbuild_protein.residues())
print(len(residues), "residues,", mbuild_protein.n_particles, "atoms, net charge", mbuild_protein.net_formal_charge)
print("ubiquitin ends, fragment begins:", [(r.name, r.resnum, r.formal_charge) for r in residues[74:80]])
print("fragment ends:", [(r.name, r.resnum, r.formal_charge) for r in residues[-3:]])

One bond record describes the modification. It is what a downstream residue library needs.

In [ ]:
bond_record, = mbuild_protein.bond_records()
bond_record

In [ ]:
from pathlib import Path

written = Path("../assets_cache/1ubq_plus_peptide.pdb")
mbuild_protein.save_pdb(written, overwrite=True)
lines = written.read_text().splitlines()
print(sorted({(line[17:20], int(line[22:26])) for line in lines if line.startswith(("ATOM", "HETATM"))}, key=lambda t: t[1])[74:82])

Pablo reads the file once the bond record is handed over as a crosslink.

In [ ]:
from openff.pablo import STD_CCD_CACHE, topology_from_pdb

pablo_residue_library = STD_CCD_CACHE.with_crosslink(
    residues=list(bond_record["residue_names"]),
    linking_atoms=list(bond_record["atom_names"]),
    leaving_atoms=[list(side) for side in bond_record["leaving_atoms"]],
    bond_order=bond_record["bond_order"],
)
openff_topology = topology_from_pdb(written, residue_library=pablo_residue_library)
print(openff_topology.n_molecules, "molecule,", openff_topology.n_atoms, "atoms,", openff_topology.n_bonds, "bonds, net charge", openff_topology.molecule(0).total_charge)
assert openff_topology.n_bonds == mbuild_protein.n_bonds